# Machine RUL Prediction — NASA C-MAPSS FD001

**Author:** Pallavi

## Data Preprocessing & Feature Preparation

In the previous notebook, I found that several sensor measurements show useful relationships with RUL, while some features remain constant.

Now I will prepare the data for regression modeling by removing features that provide no variation, separating the target from the predictors, and creating a leakage-aware validation setup.

The goal is to prepare clean and meaningful inputs for the Linear Regression model without losing the engine-level structure of the dataset.

## 1. Preparing the Training Data

I will reload the training data and create the RUL target so that the preprocessing steps are applied consistently from the original data.

In [1]:
# Importing required libraries
import pandas as pd
import numpy as np

In [2]:
# Loading the training data

columns = (
    ["unit_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(
    "../data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

# Creating the RUL target

max_cycle = train_df.groupby("unit_id")["cycle"].transform("max")
train_df["RUL"] = max_cycle - train_df["cycle"]

train_df.head()

,unit_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187


## 2. Removing Constant Features

The EDA showed that some features have no variation across the dataset. Since a constant feature cannot help the model distinguish between engine conditions, I will remove these features.

In [3]:
# Finding constant features
constant_columns = train_df.columns[train_df.nunique() == 1]
constant_columns

Index(['setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16',
       'sensor_18', 'sensor_19'],
      dtype='object')

In [4]:
# Removing constant features
train_df = train_df.drop(columns=constant_columns)
train_df.shape

(20631, 20)

## 3. Separating Features and Target

RUL is the value we want to predict, so I will separate it from the input features. The engine ID will be kept separately because it identifies the engine but should not be used as a numerical feature.

In [5]:
# Separating features and target
engine_id = train_df["unit_id"]
X = train_df.drop(columns=["unit_id", "RUL"])
y = train_df["RUL"]
print("Features:", X.shape)
print("Target:", y.shape)

Features: (20631, 18)
Target: (20631,)


## 4. Creating an Engine-Level Validation Set

Observations from the same engine form a connected degradation trajectory. Therefore, I will split the data by engine rather than by individual rows to reduce the risk of data leakage.

In [6]:
# Splitting engines for validation

from sklearn.model_selection import train_test_split
engine_ids = train_df["unit_id"].unique()
train_units, val_units = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42
)
train_data = train_df[train_df["unit_id"].isin(train_units)]
val_data = train_df[train_df["unit_id"].isin(val_units)]
print("Training engines:", train_data["unit_id"].nunique())
print("Validation engines:", val_data["unit_id"].nunique())

Training engines: 80
Validation engines: 20


## 5. Creating the Final Modeling Data

I will now separate the predictors and RUL target for the training and validation engines. These datasets will be used for the baseline regression model.

In [7]:
# Preparing training and validation data
X_train = train_data.drop(columns=["unit_id", "RUL"])
y_train = train_data["RUL"]
X_val = val_data.drop(columns=["unit_id", "RUL"])
y_val = val_data["RUL"]
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (16561, 18)
X_val: (4070, 18)
y_train: (16561,)
y_val: (4070,)


## 6. Validating the Prepared Features

Before modeling, I will verify the final feature set to ensure that only the intended predictors are included.

In [8]:
# Checking the final feature columns
print("Number of features:", len(X.columns))
print("\nFeatures:")
print(X.columns.tolist())

Number of features: 18

Features:
['cycle', 'setting_1', 'setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [9]:
# Checking for any remaining constant features
X.nunique().sort_values().head()

sensor_6      2
setting_2    13
sensor_17    13
sensor_8     53
sensor_13    56
dtype: int64

## 7. Final Preprocessing Check
Before moving to model training, I will verify that the training and validation data have the same feature structure and contain no missing values.

In [11]:
# Validating the prepared datasets
print("Same features:", list(X_train.columns) == list(X_val.columns))
print("Missing values in training:", X_train.isnull().sum().sum())
print("Missing values in validation:", X_val.isnull().sum().sum())

Same features: True
Missing values in training: 0
Missing values in validation: 0


In [ ]:
## 8. What I Learned From Preprocessing
The raw FD001 training data was converted into a modeling-ready dataset by removing constant features and separating the RUL target from the predictors.
The engine identifier was retained for grouping purposes but excluded from the model features because it is an identifier rather than a machine measurement.
Instead of randomly splitting individual observations, I created an engine-level validation set containing 20 unseen engines. This reduces the risk of leakage between related observations from the same engine.
The resulting training and validation datasets contain the same feature structure and are ready for regression modeling.
The next step is to establish a baseline Linear Regression model and evaluate its performance using the project's locked metrics: MAE, RMSE, and R².